In [ ]:
import json
import pandas as pd
import joblib

from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ----------------------------------------------------
# 1. LOAD TRAINING DATA
# ----------------------------------------------------
data = []

with open(
    r"Dataset/LeetCodeDataset-v0.3.1-train.jsonl",
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        obj = json.loads(line)

        data.append({
            "question": obj["problem_description"],
            "difficulty": obj["difficulty"],
            "tags": " ".join(obj["tags"])
        })

df_train = pd.DataFrame(data)

print("Training Samples:", len(df_train))


# ----------------------------------------------------
# 2. LOAD TESTING DATA
# ----------------------------------------------------

data = []

with open(
    r"Dataset/LeetCodeDataset-v0.3.1-test.jsonl",
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        obj = json.loads(line)

        data.append({
            "question": obj["problem_description"],
            "difficulty": obj["difficulty"],
            "tags": " ".join(obj["tags"])
        })

df_test = pd.DataFrame(data)

print("Testing Samples:", len(df_test))


# ----------------------------------------------------
# 3. ENCODE PROBLEM TAGS
# ----------------------------------------------------

train_tags = df_train["tags"].str.split()
test_tags = df_test["tags"].str.split()

mlb = MultiLabelBinarizer()

X_train_tags = mlb.fit_transform(train_tags)
X_test_tags = mlb.transform(test_tags)


# ----------------------------------------------------
# 4. PREPARE TEXT AND TARGET
# ----------------------------------------------------

X_train_text = df_train["question"]
y_train = df_train["difficulty"]

X_test_text = df_test["question"]
y_test = df_test["difficulty"]


# ----------------------------------------------------
# 5. TF-IDF VECTORIZATION
# ----------------------------------------------------

vectorizer = TfidfVectorizer(
    max_features=30000,
    stop_words="english",
    ngram_range=(1, 6),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)

X_test_tfidf = vectorizer.transform(X_test_text)


# ----------------------------------------------------
# 6. COMBINE TF-IDF + TAG FEATURES
# ----------------------------------------------------

X_train = hstack([
    X_train_tfidf,
    X_train_tags
])

X_test = hstack([
    X_test_tfidf,
    X_test_tags
])

print("Feature Shape:", X_train.shape)


# ----------------------------------------------------
# 7. MODEL 1 — LINEAR SVC
# ----------------------------------------------------

print("\nTraining LinearSVC...")

svm_model = LinearSVC(
    C=3.5,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(
    X_train,
    y_train
)

svm_predictions = svm_model.predict(
    X_test
)


# ----------------------------------------------------
# 8. MODEL 2 — LOGISTIC REGRESSION
# ----------------------------------------------------

print("Training Logistic Regression...")

lr_model = LogisticRegression(
    C=3.5,
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

lr_model.fit(
    X_train,
    y_train
)

lr_predictions = lr_model.predict(
    X_test
)


# ----------------------------------------------------
# 9. MODEL PERFORMANCE COMPARISON
# ----------------------------------------------------

models = {
    "LinearSVC": svm_predictions,
    "Logistic Regression": lr_predictions
}

results = []

for model_name, predictions in models.items():

    report = classification_report(
        y_test,
        predictions,
        output_dict=True
    )

    results.append({
        "Model": model_name,

        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "Precision": report["macro avg"]["precision"],

        "Recall": report["macro avg"]["recall"],

        "F1-Score": report["macro avg"]["f1-score"]
    })


results_df = pd.DataFrame(results)


# ----------------------------------------------------
# 10. DISPLAY PERFORMANCE TABLE
# ----------------------------------------------------

print("\n")
print("=" * 75)
print("                    MODEL PERFORMANCE")
print("=" * 75)

print(
    results_df.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-Score": "{:.4f}".format
        }
    )
)

print("=" * 75)


# ----------------------------------------------------
# 11. LINEAR SVC — DETAILED RESULTS
# ----------------------------------------------------

print("\nLinearSVC Classification Report:")
print(
    classification_report(
        y_test,
        svm_predictions
    )
)

print("LinearSVC Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        svm_predictions
    )
)


# ----------------------------------------------------
# 12. LOGISTIC REGRESSION — DETAILED RESULTS
# ----------------------------------------------------

print("\nLogistic Regression Classification Report:")
print(
    classification_report(
        y_test,
        lr_predictions
    )
)

print("Logistic Regression Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        lr_predictions
    )
)


# ----------------------------------------------------
# 13. CONFIDENCE SCORE
# ----------------------------------------------------

probabilities = lr_model.predict_proba(X_test)

confidence = probabilities.max(axis=1)

confidence_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": lr_predictions,
    "Confidence": confidence
})


print("\n")
print("=" * 60)
print("              PREDICTION CONFIDENCE")
print("=" * 60)

print(
    confidence_results.head(20).to_string(
        index=False,
        formatters={
            "Confidence": "{:.2%}".format
        }
    )
)

print("=" * 60)


# ----------------------------------------------------
# 14. AVERAGE CONFIDENCE
# ----------------------------------------------------

average_confidence = confidence.mean()

print(
    f"\nAverage Model Confidence: "
    f"{average_confidence:.2%}"
)


# ----------------------------------------------------
# 15. SAVE MODELS
# ----------------------------------------------------

joblib.dump(
    svm_model,
    "difficulty_model_svm.pkl"
)

joblib.dump(
    lr_model,
    "difficulty_model_logistic.pkl"
)

joblib.dump(
    vectorizer,
    "vectorizer.pkl"
)

joblib.dump(
    mlb,
    "tag_encoder.pkl"
)


print("\nModels Saved Successfully!")

Training Samples: 2641
Testing Samples: 228
Feature Shape: (2641, 30082)

Training LinearSVC...
Training Logistic Regression...


                    MODEL PERFORMANCE
              Model Accuracy Precision Recall F1-Score
          LinearSVC   0.6711    0.6713 0.6984   0.6774
Logistic Regression   0.6360    0.6331 0.6693   0.6422

LinearSVC Classification Report:
              precision    recall  f1-score   support

        Easy       0.66      0.88      0.75        48
        Hard       0.69      0.56      0.62        79
      Medium       0.67      0.66      0.67       101

    accuracy                           0.67       228
   macro avg       0.67      0.70      0.68       228
weighted avg       0.67      0.67      0.67       228

LinearSVC Confusion Matrix:
[[42  1  5]
 [ 7 44 28]
 [15 19 67]]

Logistic Regression Classification Report:
              precision    recall  f1-score   support

        Easy       0.59      0.83      0.69        48
        Hard       0.64      0.62 